In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip
/kaggle/input/competitions/word2vec-nlp-tutorial/sampleSubmission.csv
/kaggle/input/competitions/word2vec-nlp-tutorial/unlabeledTrainData.tsv.zip
/kaggle/input/competitions/word2vec-nlp-tutorial/labeledTrainData.tsv.zip
/kaggle/input/datasets/gongbaoxin/common-crawl-840b/glove.840B.300d.txt


In [3]:
import logging
import os
import re
import sys
import numpy as np
from itertools import chain
from gensim.models import KeyedVectors
import gensim
import pandas as pd
import torch
from bs4 import BeautifulSoup
from sklearn.model_selection import train_test_split
import pickle

# =================== 超参（和你本地保持一致） ===================
embed_size = 300
max_len = 512

# =================== Kaggle路径【自行核对修改】 ===================
TRAIN_PATH = "/kaggle/input/competitions/word2vec-nlp-tutorial/labeledTrainData.tsv.zip"
TEST_PATH = "/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip"
GLOVE_PATH = "/kaggle/input/datasets/gongbaoxin/common-crawl-840b/glove.840B.300d.txt"

# =================== 文本清洗函数（原版不动） ===================
def review_to_wordlist(review, remove_stopwords=False):
    review_text = BeautifulSoup(review, "lxml").get_text()
    review_text = re.sub("[^a-zA-Z]", " ", review_text)
    words = review_text.lower().split()
    return words

def encode_samples(tokenized_samples, word_to_idx):
    features = []
    for sample in tokenized_samples:
        feature = []
        for token in sample:
            if token in word_to_idx:
                feature.append(word_to_idx[token])
            else:
                feature.append(0)
        features.append(feature)
    return features

def pad_samples(features, maxlen=max_len, PAD=0):
    padded_features = []
    for feature in features:
        if len(feature) >= maxlen:
            padded_feature = feature[:maxlen]
        else:
            padded_feature = feature.copy()
            while len(padded_feature) < maxlen:
                padded_feature.append(PAD)
        padded_features.append(padded_feature)
    return padded_features

# =================== 主流程 ===================
os.makedirs("/kaggle/working/pickle", exist_ok=True)

train = pd.read_csv(TRAIN_PATH, header=0, delimiter="\t", quoting=3)
test = pd.read_csv(TEST_PATH, header=0, delimiter="\t", quoting=3)

clean_train_reviews, train_labels = [], []
for i, review in enumerate(train["review"]):
    clean_train_reviews.append(review_to_wordlist(review))
    train_labels.append(train["sentiment"][i])

clean_test_reviews = []
for review in test["review"]:
    clean_test_reviews.append(review_to_wordlist(review))

vocab = set(chain(*clean_train_reviews)) | set(chain(*clean_test_reviews))
vocab_size = len(vocab)

train_reviews, val_reviews, train_labels, val_labels = train_test_split(
    clean_train_reviews, train_labels, test_size=0.2, random_state=0)

# ===================【重点修改】适配Common Crawl 840B（glove-gensim分割逻辑） ===================
wvmodel = KeyedVectors(embed_size)
word_dict = {}
with open(GLOVE_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        tokens = line.split()
        if len(tokens) <= embed_size:
            continue
        try:
            vec = np.array(tokens[-embed_size:], dtype=np.float32)
            word = " ".join(tokens[:-embed_size])
            word_dict[word] = vec
        except ValueError:
            continue
wvmodel.add_vectors(list(word_dict.keys()), list(word_dict.values()))
print(f"GloVe加载完成，载入词总数：{len(wvmodel)}")
# =========================================================================================

word_to_idx = {word: i + 1 for i, word in enumerate(vocab)}
word_to_idx['<unk>'] = 0
idx_to_word = {i + 1: word for i, word in enumerate(vocab)}
idx_to_word[0] = '<unk>'

train_features = torch.tensor(pad_samples(encode_samples(train_reviews, word_to_idx)))
val_features = torch.tensor(pad_samples(encode_samples(val_reviews, word_to_idx)))
test_features = torch.tensor(pad_samples(encode_samples(clean_test_reviews, word_to_idx)))

train_labels = torch.tensor(train_labels)
val_labels = torch.tensor(val_labels)

# 构建Embedding权重矩阵
weight = torch.zeros(vocab_size + 1, embed_size)
hit = 0
for word, idx in word_to_idx.items():
    if word in wvmodel:
        weight[idx, :] = torch.from_numpy(wvmodel.get_vector(word))
        hit += 1
print(f"词表匹配成功向量：{hit}/{len(word_to_idx)}")

pickle_file = "/kaggle/working/pickle/imdb_glove.pickle3"
pickle.dump(
    [train_features, train_labels, val_features, val_labels, test_features, weight, word_to_idx, idx_to_word, vocab],
    open(pickle_file, 'wb'))
print('pickle文件生成完成！')

GloVe加载完成，载入词总数：2195895


/tmp/ipykernel_58/1117333043.py:114: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  weight[idx, :] = torch.from_numpy(wvmodel.get_vector(word))


词表匹配成功向量：77554/101400
pickle文件生成完成！


In [6]:
import logging
import os
import sys
import pickle
import time

import pandas as pd
import torch
from torch import nn
from torch import optim
from tqdm import tqdm
from sklearn.metrics import accuracy_score

test = pd.read_csv("/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip", header=0, delimiter="\t", quoting=3)

num_epochs = 10
embed_size = 300
num_hiddens = 120
num_layers = 2
bidirectional = True
batch_size = 64
labels = 2
lr = 0.001
dropout_rate = 0.25

# 强制GPU模式
if not torch.cuda.is_available():
    raise RuntimeError("当前环境无可用GPU，请切换GPU运行时！")
device = torch.device("cuda")
use_gpu = True
print(f"使用GPU设备: {torch.cuda.get_device_name(device)}")

class SentimentNet(nn.Module):
    def __init__(self, embed_size, num_hiddens, num_layers, bidirectional, weight, labels, use_gpu, dropout=0.25, **kwargs):
        super(SentimentNet, self).__init__(**kwargs)
        self.num_hiddens = num_hiddens
        self.num_layers = num_layers
        self.use_gpu = use_gpu
        self.bidirectional = bidirectional

        self.embedding = nn.Embedding.from_pretrained(weight)
        # 关键：冻结词向量，防止微调带来分数暴跌
        self.embedding.weight.requires_grad = False

        self.encoder = nn.LSTM(input_size=embed_size, hidden_size=self.num_hiddens,
                               num_layers=num_layers, bidirectional=self.bidirectional,
                               dropout=dropout if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)

        if self.bidirectional:
            self.decoder = nn.Linear(num_hiddens * 4, labels)
        else:
            self.decoder = nn.Linear(num_hiddens * 2, labels)

    def forward(self, inputs):
        embeddings = self.embedding(inputs)
        states, hidden = self.encoder(embeddings.permute([1, 0, 2]))
        encoding = torch.cat([states[0], states[-1]], dim=1)
        encoding = self.dropout(encoding)
        outputs = self.decoder(encoding)
        return outputs

if __name__ == '__main__':
    program = os.path.basename(sys.argv[0])
    logger = logging.getLogger(program)
    logging.basicConfig(format='%(asctime)s: %(levelname)s: %(message)s')
    logging.root.setLevel(logging.INFO)
    logger.info(r"running %s" % ''.join(sys.argv))

    logging.info('loading data...')
    pickle_file = os.path.join('pickle', 'imdb_glove.pickle3')
    [train_features, train_labels, val_features, val_labels, test_features, weight, word_to_idx, idx_to_word,
     vocab] = pickle.load(open(pickle_file, 'rb'))
    logging.info('data loaded!')

    weight = weight.to(device)
    net = SentimentNet(embed_size=embed_size, num_hiddens=num_hiddens, num_layers=num_layers,
                       bidirectional=bidirectional, weight=weight,
                       labels=labels, use_gpu=use_gpu, dropout=dropout_rate)
    net.to(device)

    # 类别权重：从轻起步，优先 [1.15, 1.0]，效果不足再上调到1.2
    class_weight = torch.tensor([1.15, 1.0]).to(device)
    loss_function = nn.CrossEntropyLoss(weight=class_weight)

    optimizer = optim.Adam(net.parameters(), lr=lr)
    # 先注释学习率调度，如果稳定收敛再启用
    # scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.8)

    train_set = torch.utils.data.TensorDataset(train_features, train_labels)
    val_set = torch.utils.data.TensorDataset(val_features, val_labels)
    test_set = torch.utils.data.TensorDataset(test_features, )

    train_iter = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_iter = torch.utils.data.DataLoader(val_set, batch_size=batch_size, shuffle=False)
    test_iter = torch.utils.data.DataLoader(test_set, batch_size=batch_size, shuffle=False)

    for epoch in range(num_epochs):
        start = time.time()
        train_loss, val_losses = 0, 0
        train_acc, val_acc = 0, 0
        n, m = 0, 0
        error_cases = []
        net.train()
        with tqdm(total=len(train_iter), desc='Epoch %d' % epoch) as pbar:
            for feature, label in train_iter:
                n += 1
                net.zero_grad()
                feature = feature.to(device)
                label = label.to(device)
                score = net(feature)
                loss = loss_function(score, label)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(net.parameters(), max_norm=5.0)
                optimizer.step()
                train_acc += accuracy_score(torch.argmax(score.data, dim=1).cpu(), label.cpu())
                train_loss += loss

                pbar.set_postfix({
                    'train loss': '%.4f' % (train_loss.data / n),
                    'train acc': '%.2f' % (train_acc / n)
                })
                pbar.update(1)

            net.eval()
            with torch.no_grad():
                for val_feature, val_label in val_iter:
                    m += 1
                    val_feature = val_feature.to(device)
                    val_label = val_label.to(device)
                    val_score = net(val_feature)
                    val_loss = loss_function(val_score, val_label)
                    pred = torch.argmax(val_score.data, dim=1)
                    val_acc += accuracy_score(pred.cpu(), val_label.cpu())
                    val_losses += val_loss

                    wrong_mask = pred != val_label
                    wrong_indices = torch.where(wrong_mask)[0]
                    for idx in wrong_indices:
                        feat_seq = val_feature[idx].cpu().tolist()
                        words = []
                        for wid in feat_seq:
                            if wid == 0:
                                continue
                            words.append(idx_to_word.get(wid, "<unk>"))
                        raw_sentence = " ".join(words)
                        error_cases.append({
                            "sentence": raw_sentence,
                            "true_label": val_label[idx].item(),
                            "pred_label": pred[idx].item()
                        })

            # scheduler.step()
            end = time.time()
            runtime = end - start
            pbar.set_postfix({
                'train loss': '%.4f' % (train_loss.data / n),
                'train acc': '%.2f' % (train_acc / n),
                'val loss': '%.4f' % (val_losses.data / m),
                'val acc': '%.2f' % (val_acc / m),
                'time': '%.2f' % runtime
            })

    print("\n==================== 3条预测错误案例（验证集）====================")
    show_cnt = min(3, len(error_cases))
    for i in range(show_cnt):
        case = error_cases[i]
        print(f"\n【案例{i+1}】")
        print(f"真实标签：{case['true_label']}，预测标签：{case['pred_label']}")
        print(f"文本内容：{case['sentence']}")

    test_pred = []
    net.eval()
    with torch.no_grad():
        with tqdm(total=len(test_iter), desc='Prediction') as pbar:
            for test_feature, in test_iter:
                test_feature = test_feature.to(device)
                test_score = net(test_feature)
                test_pred.extend(torch.argmax(test_score.data, dim=1).cpu().numpy().tolist())
                pbar.update(1)

    os.makedirs("./result", exist_ok=True)
    result_output = pd.DataFrame(data={"id": test["id"], "sentiment": test_pred})
    result_output.to_csv("./result/lstm_improved.csv", index=False, quoting=3)
    logging.info('result saved!')

2026-08-14 23:37:28,951: INFO: running /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py-f/root/.local/share/jupyter/runtime/kernel-8ddd64ca-b7e2-40fd-827a-5ba318d35b50.json
2026-08-14 23:37:28,952: INFO: loading data...


使用GPU设备: Tesla T4


2026-08-14 23:37:29,475: INFO: data loaded!
Epoch 9: 100%|██████████| 313/313 [00:27<00:00, 11.29it/s, train loss=0.2160, train acc=0.92, val loss=0.3018, val acc=0.88, time=27.72]



==================== 3条预测错误案例（验证集）====================

【案例1】
真实标签：0，预测标签：1
文本内容：natural born killers cinema cut r director s cut nc it s an unusual oliver stone picture but when i read he was on drugs during the filming i needed no further explanation natural born killers is a risky mad all out film making that we do not get very often strange psychotic artistic pictures natural born killers is basically the story of how two mass killers were popularised and glorified by the media there is a great scene where an interviewer questions some teenagers about mickey and mallory and the teenager says murder is wrong but if i was a mass murderer i d be mickey and mallory mickey describes this with a situation of frankenstein the monster and dr frankenstein dr frankenstein is the media who has turned them into these monstrous killersmost oliver stone films examine the flaws of the america the country that the director loves and admires i guess natural born killers is about the effect of mass

Prediction: 100%|██████████| 391/391 [00:10<00:00, 38.82it/s]
2026-08-14 23:42:22,674: INFO: result saved!


In [7]:
import logging
import os
import sys
import pickle
import time

import pandas as pd
import torch
from torch import nn
from torch import optim
from tqdm import tqdm
from sklearn.metrics import accuracy_score

test = pd.read_csv("/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip", header=0, delimiter="\t", quoting=3)

num_epochs = 10
embed_size = 300
num_hiddens = 120
num_layers = 2
bidirectional = True
batch_size = 64
labels = 2
lr = 0.001

# 强制GPU模式，没有GPU直接终止程序
if not torch.cuda.is_available():
    raise RuntimeError("当前环境无可用GPU，请切换GPU运行时！")
device = torch.device("cuda")
use_gpu = True
print(f"使用GPU设备: {torch.cuda.get_device_name(device)}")

class SentimentNet(nn.Module):
    def __init__(self, embed_size, num_hiddens, num_layers, bidirectional, weight, labels, use_gpu, **kwargs):
        super(SentimentNet, self).__init__(**kwargs)
        self.num_hiddens = num_hiddens
        self.num_layers = num_layers
        self.use_gpu = use_gpu
        self.bidirectional = bidirectional
        self.embedding = nn.Embedding.from_pretrained(weight)
        self.embedding.weight.requires_grad = False
        self.encoder = nn.LSTM(input_size=embed_size, hidden_size=self.num_hiddens,
                               num_layers=num_layers, bidirectional=self.bidirectional,
                               dropout=0)
        if self.bidirectional:
            self.decoder = nn.Linear(num_hiddens * 4, labels)
        else:
            self.decoder = nn.Linear(num_hiddens * 2, labels)

    def forward(self, inputs):
        embeddings = self.embedding(inputs)
        states, hidden = self.encoder(embeddings.permute([1, 0, 2]))
        encoding = torch.cat([states[0], states[-1]], dim=1)
        outputs = self.decoder(encoding)
        return outputs

if __name__ == '__main__':
    program = os.path.basename(sys.argv[0])
    logger = logging.getLogger(program)

    logging.basicConfig(format='%(asctime)s: %(levelname)s: %(message)s')
    logging.root.setLevel(logging.INFO)
    logger.info(r"running %s" % ''.join(sys.argv))

    logging.info('loading data...')
    pickle_file = os.path.join('pickle', 'imdb_glove.pickle3')
    [train_features, train_labels, val_features, val_labels, test_features, weight, word_to_idx, idx_to_word,
     vocab] = pickle.load(open(pickle_file, 'rb'))
    logging.info('data loaded!')

    weight = weight.to(device)
    net = SentimentNet(embed_size=embed_size, num_hiddens=num_layers, num_layers=num_layers,
                       bidirectional=bidirectional, weight=weight,
                       labels=labels, use_gpu=use_gpu)
    net.to(device)

    # =========关键：先使用原始无权重损失，如果效果下滑立刻换回下面注释版本=========
    loss_function = nn.CrossEntropyLoss()
    # 如果验证集大量0→1误判，可以放开下面一行尝试，一旦分数下降立刻注释
    # class_weight = torch.tensor([1.1, 1.0]).to(device)
    # loss_function = nn.CrossEntropyLoss(weight=class_weight)

    optimizer = optim.Adam(net.parameters(), lr=lr)

    train_set = torch.utils.data.TensorDataset(train_features, train_labels)
    val_set = torch.utils.data.TensorDataset(val_features, val_labels)
    test_set = torch.utils.data.TensorDataset(test_features, )

    train_iter = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_iter = torch.utils.data.DataLoader(val_set, batch_size=batch_size, shuffle=False)
    test_iter = torch.utils.data.DataLoader(test_set, batch_size=batch_size, shuffle=False)

    for epoch in range(num_epochs):
        start = time.time()
        train_loss, val_losses = 0, 0
        train_acc, val_acc = 0, 0
        n, m = 0, 0
        error_cases = []   # 每轮清空错误样本
        net.train()
        with tqdm(total=len(train_iter), desc='Epoch %d' % epoch) as pbar:
            for feature, label in train_iter:
                n += 1
                net.zero_grad()
                feature = feature.to(device)
                label = label.to(device)
                score = net(feature)
                loss = loss_function(score, label)
                loss.backward()
                optimizer.step()
                train_acc += accuracy_score(torch.argmax(score.data, dim=1).cpu(), label.cpu())
                train_loss += loss

                pbar.set_postfix({
                    'train loss': '%.4f' % (train_loss.data / n),
                    'train acc': '%.2f' % (train_acc / n)
                })
                pbar.update(1)

            net.eval()
            with torch.no_grad():
                for val_feature, val_label in val_iter:
                    m += 1
                    val_feature = val_feature.to(device)
                    val_label = val_label.to(device)
                    val_score = net(val_feature)
                    val_loss = loss_function(val_score, val_label)
                    pred = torch.argmax(val_score.data, dim=1)
                    val_acc += accuracy_score(pred.cpu(), val_label.cpu())
                    val_losses += val_loss

                    wrong_mask = pred != val_label
                    wrong_indices = torch.where(wrong_mask)[0]
                    for idx in wrong_indices:
                        feat_seq = val_feature[idx].cpu().tolist()
                        words = []
                        for wid in feat_seq:
                            if wid == 0:
                                continue
                            words.append(idx_to_word.get(wid, "<unk>"))
                        raw_sentence = " ".join(words)
                        error_cases.append({
                            "sentence": raw_sentence,
                            "true_label": val_label[idx].item(),
                            "pred_label": pred[idx].item()
                        })

            end = time.time()
            runtime = end - start
            pbar.set_postfix({
                'train loss': '%.4f' % (train_loss.data / n),
                'train acc': '%.2f' % (train_acc / n),
                'val loss': '%.4f' % (val_losses.data / m),
                'val acc': '%.2f' % (val_acc / m),
                'time': '%.2f' % runtime
            })

    print("\n==================== 3条预测错误案例（验证集）====================")
    show_cnt = min(3, len(error_cases))
    for i in range(show_cnt):
        case = error_cases[i]
        print(f"\n【案例{i+1}】")
        print(f"真实标签：{case['true_label']}，预测标签：{case['pred_label']}")
        print(f"文本内容：{case['sentence']}")

    test_pred = []
    net.eval()
    with torch.no_grad():
        with tqdm(total=len(test_iter), desc='Prediction') as pbar:
            for test_feature, in test_iter:
                test_feature = test_feature.to(device)
                test_score = net(test_feature)
                test_pred.extend(torch.argmax(test_score.data, dim=1).cpu().numpy().tolist())
                pbar.update(1)

    os.makedirs("./result", exist_ok=True)
    result_output = pd.DataFrame(data={"id": test["id"], "sentiment": test_pred})
    result_output.to_csv("./result/lstm.csv", index=False, quoting=3)
    logging.info('result saved!')

2026-08-14 23:43:51,630: INFO: running /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py-f/root/.local/share/jupyter/runtime/kernel-8ddd64ca-b7e2-40fd-827a-5ba318d35b50.json
2026-08-14 23:43:51,631: INFO: loading data...


使用GPU设备: Tesla T4


2026-08-14 23:43:52,122: INFO: data loaded!
Epoch 9: 100%|██████████| 313/313 [00:03<00:00, 96.21it/s, train loss=0.5264, train acc=0.75, val loss=0.5225, val acc=0.75, time=3.26] 



==================== 3条预测错误案例（验证集）====================

【案例1】
真实标签：0，预测标签：1
文本内容：i vaguely remember ben from my sci fi fandom days of the s i was doing several interviews bios of obscure actors actresses most notably ben actress fay spain and jody fair who played angela in s the young savages ben was one of the people at a low key sci fi con in chicago about when i had a nice chat with him and his career and life all these were published in some now long forgotten fanzine of the day wish i still had copies of those interviews but time marches on and any of those people surely wouldn t remember me at all so many years later ben was a really nice fellow ekeing out a living the cons of those days didn t even pay their guest unless of course they were big name stars and even then the pay was a couple hundred dollars at most good to know ben s still alive kicking how bout a remake of creature but years older ugly then uglier now

【案例2】
真实标签：0，预测标签：1
文本内容：this santa movie starts off strange

Prediction: 100%|██████████| 391/391 [00:01<00:00, 367.18it/s]
2026-08-14 23:44:26,898: INFO: result saved!


In [9]:
import logging
import os
import sys
import pickle
import time

import pandas as pd
import torch
from torch import nn
from torch import optim
from tqdm import tqdm
from sklearn.metrics import accuracy_score

test = pd.read_csv("/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip", header=0, delimiter="\t", quoting=3)

num_epochs = 10
embed_size = 300
num_hiddens = 120
num_layers = 2
bidirectional = True
batch_size = 64
labels = 2
lr = 0.001

# 强制GPU模式，没有GPU直接终止程序
if not torch.cuda.is_available():
    raise RuntimeError("当前环境无可用GPU，请切换GPU运行时！")
device = torch.device("cuda")
use_gpu = True
print(f"使用GPU设备: {torch.cuda.get_device_name(device)}")

class SentimentNet(nn.Module):
    def __init__(self, embed_size, num_hiddens, num_layers, bidirectional, weight, labels, use_gpu, **kwargs):
        super(SentimentNet, self).__init__(**kwargs)
        self.num_hiddens = num_hiddens
        self.num_layers = num_layers
        self.use_gpu = use_gpu
        self.bidirectional = bidirectional

        self.embedding = nn.Embedding.from_pretrained(weight)
        self.embedding.weight.requires_grad = False

        self.encoder = nn.LSTM(input_size=embed_size,
                               hidden_size=self.num_hiddens,
                               num_layers=num_layers,
                               bidirectional=self.bidirectional,
                               dropout=0)

        # ============= 修正维度 =============
        # 双向：last_cat=240，max_pool=240，总和=480 = num_hiddens * 4
        # 单向：last_cat=120，max_pool=120，总和=240 = num_hiddens * 2
        if self.bidirectional:
            self.decoder = nn.Linear(num_hiddens * 4, labels)
        else:
            self.decoder = nn.Linear(num_hiddens * 2, labels)

    def forward(self, inputs):
        embeddings = self.embedding(inputs)
        states, hidden = self.encoder(embeddings.permute([1, 0, 2]))

        # 双向LSTM最后一层的前向/反向最后时刻状态
        forward_last = states[-1, :, :self.num_hiddens]
        backward_last = states[0, :, self.num_hiddens:]
        last_cat = torch.cat([forward_last, backward_last], dim=-1)

        # 全局最大池：捕捉整句中最强情感信号
        max_pool = torch.max(states, dim=0)[0]

        encoding = torch.cat([last_cat, max_pool], dim=-1)
        outputs = self.decoder(encoding)

        return outputs

if __name__ == '__main__':
    program = os.path.basename(sys.argv[0])
    logger = logging.getLogger(program)

    logging.basicConfig(format='%(asctime)s: %(levelname)s: %(message)s')
    logging.root.setLevel(logging.INFO)
    logger.info(r"running %s" % ''.join(sys.argv))

    logging.info('loading data...')
    pickle_file = os.path.join('pickle', 'imdb_glove.pickle3')
    [train_features, train_labels, val_features, val_labels, test_features, weight, word_to_idx, idx_to_word,
     vocab] = pickle.load(open(pickle_file, 'rb'))
    logging.info('data loaded!')

    weight = weight.to(device)
    net = SentimentNet(embed_size=embed_size,
                       num_hiddens=num_hiddens,
                       num_layers=num_layers,
                       bidirectional=bidirectional,
                       weight=weight,
                       labels=labels,
                       use_gpu=use_gpu)
    net.to(device)

    loss_function = nn.CrossEntropyLoss()
    optimizer = optim.Adam(net.parameters(), lr=lr)

    train_set = torch.utils.data.TensorDataset(train_features, train_labels)
    val_set = torch.utils.data.TensorDataset(val_features, val_labels)
    test_set = torch.utils.data.TensorDataset(test_features, )

    train_iter = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_iter = torch.utils.data.DataLoader(val_set, batch_size=batch_size, shuffle=False)
    test_iter = torch.utils.data.DataLoader(test_set, batch_size=batch_size, shuffle=False)

    for epoch in range(num_epochs):
        start = time.time()
        train_loss, val_losses = 0, 0
        train_acc, val_acc = 0, 0
        n, m = 0, 0
        error_cases = []

        net.train()
        with tqdm(total=len(train_iter), desc='Epoch %d' % epoch) as pbar:
            for feature, label in train_iter:
                n += 1
                net.zero_grad()
                feature = feature.to(device)
                label = label.to(device)

                score = net(feature)
                loss = loss_function(score, label)
                loss.backward()
                optimizer.step()

                train_acc += accuracy_score(torch.argmax(score.data, dim=1).cpu(), label.cpu())
                train_loss += loss

                pbar.set_postfix({
                    'train loss': '%.4f' % (train_loss.data / n),
                    'train acc': '%.2f' % (train_acc / n)
                })
                pbar.update(1)

        net.eval()
        with torch.no_grad():
            for val_feature, val_label in val_iter:
                m += 1
                val_feature = val_feature.to(device)
                val_label = val_label.to(device)

                val_score = net(val_feature)
                val_loss = loss_function(val_score, val_label)
                pred = torch.argmax(val_score.data, dim=1)

                val_acc += accuracy_score(pred.cpu(), val_label.cpu())
                val_losses += val_loss

                wrong_mask = pred != val_label
                wrong_indices = torch.where(wrong_mask)[0]

                for idx in wrong_indices:
                    feat_seq = val_feature[idx].cpu().tolist()
                    words = []
                    for wid in feat_seq:
                        if wid == 0:
                            continue
                        words.append(idx_to_word.get(wid, "<unk>"))
                    raw_sentence = " ".join(words)
                    error_cases.append({
                        "sentence": raw_sentence,
                        "true_label": val_label[idx].item(),
                        "pred_label": pred[idx].item()
                    })

        end = time.time()
        runtime = end - start
        pbar.set_postfix({
            'train loss': '%.4f' % (train_loss.data / n),
            'train acc': '%.2f' % (train_acc / n),
            'val loss': '%.4f' % (val_losses.data / m),
            'val acc': '%.2f' % (val_acc / m),
            'time': '%.2f' % runtime
        })

    print("\n==================== 3条预测错误案例（验证集）====================")
    show_cnt = min(3, len(error_cases))
    for i in range(show_cnt):
        case = error_cases[i]
        print(f"\n【案例{i+1}】")
        print(f"真实标签：{case['true_label']}，预测标签：{case['pred_label']}")
        print(f"文本内容：{case['sentence']}")

    test_pred = []
    net.eval()
    with torch.no_grad():
        with tqdm(total=len(test_iter), desc='Prediction') as pbar:
            for test_feature, in test_iter:
                test_feature = test_feature.to(device)
                test_score = net(test_feature)
                test_pred.extend(torch.argmax(test_score.data, dim=1).cpu().numpy().tolist())
                pbar.update(1)

    os.makedirs("./result", exist_ok=True)
    result_output = pd.DataFrame(data={"id": test["id"], "sentiment": test_pred})
    result_output.to_csv("./result/lstm.csv", index=False, quoting=3)
    logging.info('result saved!')


2026-08-14 23:49:47,199: INFO: running /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py-f/root/.local/share/jupyter/runtime/kernel-8ddd64ca-b7e2-40fd-827a-5ba318d35b50.json
2026-08-14 23:49:47,200: INFO: loading data...


使用GPU设备: Tesla T4


2026-08-14 23:49:47,693: INFO: data loaded!
Epoch 9: 100%|██████████| 313/313 [00:25<00:00, 12.34it/s, train loss=0.0152, train acc=1.00]



==================== 3条预测错误案例（验证集）====================

【案例1】
真实标签：0，预测标签：1
文本内容：i vaguely remember ben from my sci fi fandom days of the s i was doing several interviews bios of obscure actors actresses most notably ben actress fay spain and jody fair who played angela in s the young savages ben was one of the people at a low key sci fi con in chicago about when i had a nice chat with him and his career and life all these were published in some now long forgotten fanzine of the day wish i still had copies of those interviews but time marches on and any of those people surely wouldn t remember me at all so many years later ben was a really nice fellow ekeing out a living the cons of those days didn t even pay their guest unless of course they were big name stars and even then the pay was a couple hundred dollars at most good to know ben s still alive kicking how bout a remake of creature but years older ugly then uglier now

【案例2】
真实标签：1，预测标签：0
文本内容：very good except for the ending whi

Prediction: 100%|██████████| 391/391 [00:10<00:00, 38.67it/s]
2026-08-14 23:54:37,460: INFO: result saved!


In [10]:
import logging
import os
import sys
import pickle
import time

import pandas as pd
import torch
from torch import nn
from torch import optim
from tqdm import tqdm
from sklearn.metrics import accuracy_score

test = pd.read_csv("/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip", header=0, delimiter="\t", quoting=3)

num_epochs = 10
embed_size = 300
num_hiddens = 120
num_layers = 2
bidirectional = True
batch_size = 64
labels = 2
lr = 0.001

# 强制GPU模式，没有GPU直接终止程序
if not torch.cuda.is_available():
    raise RuntimeError("当前环境无可用GPU，请切换GPU运行时！")
device = torch.device("cuda")
use_gpu = True
print(f"使用GPU设备: {torch.cuda.get_device_name(device)}")

class SentimentNet(nn.Module):
    def __init__(self, embed_size, num_hiddens, num_layers, bidirectional, weight, labels, use_gpu, **kwargs):
        super(SentimentNet, self).__init__(**kwargs)
        self.num_hiddens = num_hiddens
        self.num_layers = num_layers
        self.use_gpu = use_gpu
        self.bidirectional = bidirectional
        self.embedding = nn.Embedding.from_pretrained(weight)
        self.embedding.weight.requires_grad = False
        self.encoder = nn.LSTM(input_size=embed_size, hidden_size=self.num_hiddens,
                               num_layers=num_layers, bidirectional=self.bidirectional,
                               dropout=0)
        
        # 双向：首尾(240) + max_pool(240) + mean_pool(240) = 720 = num_hiddens * 6
        if self.bidirectional:
            feature_dim = num_hiddens * 6
        else:
            feature_dim = num_hiddens * 3
        
        # 增加BN+Dropout，弱化局部极端词语干扰
        self.bn = nn.BatchNorm1d(feature_dim)
        self.dropout = nn.Dropout(0.25)
        self.decoder = nn.Linear(feature_dim, labels)

    def forward(self, inputs):
        embeddings = self.embedding(inputs)
        states, hidden = self.encoder(embeddings.permute([1, 0, 2]))
        
        forward_last = states[-1, :, :self.num_hiddens]
        backward_last = states[0, :, self.num_hiddens:]
        last_cat = torch.cat([forward_last, backward_last], dim=-1)
        
        max_pool = torch.max(states, dim=0)[0]
        mean_pool = torch.mean(states, dim=0)
        
        encoding = torch.cat([last_cat, max_pool, mean_pool], dim=-1)
        encoding = self.bn(encoding)
        encoding = self.dropout(encoding)
        outputs = self.decoder(encoding)
        return outputs

if __name__ == '__main__':
    program = os.path.basename(sys.argv[0])
    logger = logging.getLogger(program)

    logging.basicConfig(format='%(asctime)s: %(levelname)s: %(message)s')
    logging.root.setLevel(logging.INFO)
    logger.info(r"running %s" % ''.join(sys.argv))

    logging.info('loading data...')
    pickle_file = os.path.join('pickle', 'imdb_glove.pickle3')
    [train_features, train_labels, val_features, val_labels, test_features, weight, word_to_idx, idx_to_word,
     vocab] = pickle.load(open(pickle_file, 'rb'))
    logging.info('data loaded!')

    weight = weight.to(device)
    net = SentimentNet(embed_size=embed_size, num_hiddens=num_hiddens, num_layers=num_layers,
                       bidirectional=bidirectional, weight=weight,
                       labels=labels, use_gpu=use_gpu)
    net.to(device)

    loss_function = nn.CrossEntropyLoss()
    optimizer = optim.Adam(net.parameters(), lr=lr)

    train_set = torch.utils.data.TensorDataset(train_features, train_labels)
    val_set = torch.utils.data.TensorDataset(val_features, val_labels)
    test_set = torch.utils.data.TensorDataset(test_features, )

    train_iter = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_iter = torch.utils.data.DataLoader(val_set, batch_size=batch_size, shuffle=False)
    test_iter = torch.utils.data.DataLoader(test_set, batch_size=batch_size, shuffle=False)

    for epoch in range(num_epochs):
        start = time.time()
        train_loss, val_losses = 0, 0
        train_acc, val_acc = 0, 0
        n, m = 0, 0
        error_cases = []
        net.train()
        with tqdm(total=len(train_iter), desc='Epoch %d' % epoch) as pbar:
            for feature, label in train_iter:
                n += 1
                net.zero_grad()
                feature = feature.to(device)
                label = label.to(device)
                score = net(feature)
                loss = loss_function(score, label)
                loss.backward()
                optimizer.step()
                train_acc += accuracy_score(torch.argmax(score.data, dim=1).cpu(), label.cpu())
                train_loss += loss

                pbar.set_postfix({
                    'train loss': '%.4f' % (train_loss.data / n),
                    'train acc': '%.2f' % (train_acc / n)
                })
                pbar.update(1)

            net.eval()
            with torch.no_grad():
                for val_feature, val_label in val_iter:
                    m += 1
                    val_feature = val_feature.to(device)
                    val_label = val_label.to(device)
                    val_score = net(val_feature)
                    val_loss = loss_function(val_score, val_label)
                    pred = torch.argmax(val_score.data, dim=1)
                    val_acc += accuracy_score(pred.cpu(), val_label.cpu())
                    val_losses += val_loss

                    wrong_mask = pred != val_label
                    wrong_indices = torch.where(wrong_mask)[0]
                    for idx in wrong_indices:
                        feat_seq = val_feature[idx].cpu().tolist()
                        words = []
                        for wid in feat_seq:
                            if wid == 0:
                                continue
                            words.append(idx_to_word.get(wid, "<unk>"))
                        raw_sentence = " ".join(words)
                        error_cases.append({
                            "sentence": raw_sentence,
                            "true_label": val_label[idx].item(),
                            "pred_label": pred[idx].item()
                        })

            end = time.time()
            runtime = end - start
            pbar.set_postfix({
                'train loss': '%.4f' % (train_loss.data / n),
                'train acc': '%.2f' % (train_acc / n),
                'val loss': '%.4f' % (val_losses.data / m),
                'val acc': '%.2f' % (val_acc / m),
                'time': '%.2f' % runtime
            })

    print("\n==================== 3条预测错误案例（验证集）====================")
    show_cnt = min(3, len(error_cases))
    for i in range(show_cnt):
        case = error_cases[i]
        print(f"\n【案例{i+1}】")
        print(f"真实标签：{case['true_label']}，预测标签：{case['pred_label']}")
        print(f"文本内容：{case['sentence']}")

    test_pred = []
    net.eval()
    with torch.no_grad():
        with tqdm(total=len(test_iter), desc='Prediction') as pbar:
            for test_feature, in test_iter:
                test_feature = test_feature.to(device)
                test_score = net(test_feature)
                test_pred.extend(torch.argmax(test_score.data, dim=1).cpu().numpy().tolist())
                pbar.update(1)

    os.makedirs("./result", exist_ok=True)
    result_output = pd.DataFrame(data={"id": test["id"], "sentiment": test_pred})
    result_output.to_csv("./result/lstm_3.csv", index=False, quoting=3)
    logging.info('result saved!')

2026-08-14 23:57:14,479: INFO: running /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py-f/root/.local/share/jupyter/runtime/kernel-8ddd64ca-b7e2-40fd-827a-5ba318d35b50.json
2026-08-14 23:57:14,479: INFO: loading data...


使用GPU设备: Tesla T4


2026-08-14 23:57:14,967: INFO: data loaded!
Epoch 9: 100%|██████████| 313/313 [00:27<00:00, 11.20it/s, train loss=0.0620, train acc=0.98, val loss=0.3453, val acc=0.90, time=27.97]



==================== 3条预测错误案例（验证集）====================

【案例1】
真实标签：0，预测标签：1
文本内容：natural born killers cinema cut r director s cut nc it s an unusual oliver stone picture but when i read he was on drugs during the filming i needed no further explanation natural born killers is a risky mad all out film making that we do not get very often strange psychotic artistic pictures natural born killers is basically the story of how two mass killers were popularised and glorified by the media there is a great scene where an interviewer questions some teenagers about mickey and mallory and the teenager says murder is wrong but if i was a mass murderer i d be mickey and mallory mickey describes this with a situation of frankenstein the monster and dr frankenstein dr frankenstein is the media who has turned them into these monstrous killersmost oliver stone films examine the flaws of the america the country that the director loves and admires i guess natural born killers is about the effect of mass

Prediction: 100%|██████████| 391/391 [00:10<00:00, 38.48it/s]
2026-08-15 00:02:10,060: INFO: result saved!
